# Month 3 — Cross-Dataset External Validation: Cleveland vs. Framingham

This notebook implements Month 3 of the research framework:
1. **Data Harmonization**: Aligning the UCI Cleveland dataset and the Framingham Heart Study dataset onto a unified 5-feature schema (`age`, `sex`, `sysBP`, `totChol`, `diabetes`, target `target`). Documenting feature loss and dataset differences.
2. **External Validation Pipeline**: Training the full tuned ensemble framework on the complete harmonized Cleveland dataset, then evaluating performance directly on the unseen complete Framingham dataset (4,240 instances).
3. **Performance Transferability Comparison**: Comparing Cleveland-only 5-fold outer nested CV performance against Framingham external validation metrics to quantify cross-dataset performance decay and generalization.


In [1]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Ensure src module is in python path
sys.path.append('..')

from src.preprocessing import load_data
from src.data_harmonization import load_framingham_raw, harmonize_datasets
from src.external_validation import run_cross_dataset_validation

# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")


Configuration loaded successfully.


## 1. Load and Harmonize Cleveland & Framingham Datasets
Load raw datasets, align schema onto common attributes (`age`, `sex`, `sysBP`, `totChol`, `diabetes`), and binarize target variables.


In [2]:
raw_cleveland_path = '../' + config['dataset']['raw_path']
df_clev_raw = load_data(raw_cleveland_path)
df_fram_raw = load_framingham_raw('../data/raw_framingham.csv')

print(f"Raw Cleveland Dataset Shape: {df_clev_raw.shape}")
print(f"Raw Framingham Dataset Shape: {df_fram_raw.shape}")

c_harm, f_harm = harmonize_datasets(df_clev_raw, df_fram_raw)

print("\nHarmonized Cleveland Shape:", c_harm.shape)
print("Harmonized Framingham Shape:", f_harm.shape)

print("\nHarmonized Schema Columns:", c_harm.columns.tolist())


Raw Cleveland Dataset Shape: (303, 14)
Raw Framingham Dataset Shape: (4240, 16)

Harmonized Cleveland Shape: (303, 6)
Harmonized Framingham Shape: (4240, 6)

Harmonized Schema Columns: ['age', 'sex', 'sysBP', 'totChol', 'diabetes', 'target']


### Documentation of Feature Loss and Simplification
- **Features dropped from Cleveland** (missing in Framingham): `cp` (chest pain), `restecg`, `thalach` (max heart rate), `exang`, `oldpeak`, `slope`, `ca` (vessels), `thal`.
- **Features dropped from Framingham** (missing in Cleveland): `education`, `currentSmoker`, `cigsPerDay`, `BPMeds`, `prevalentStroke`, `prevalentHyp`, `diaBP`, `BMI`, `heartRate`, `glucose`.
- **Target Definitions**:
  - *Cleveland*: Angiographic disease status (`target > 0`).
  - *Framingham*: 10-year risk of coronary heart disease (`TenYearCHD`).


## 2. Execute Cross-Dataset External Validation
Train tuned classifiers (RF, XGBoost, AdaBoost, Soft Ensemble) on full Cleveland harmonized dataset and evaluate on the unseen Framingham dataset. Also compute baseline Cleveland nested CV on the harmonized feature set.


In [3]:
print("Executing cross-dataset validation pipeline...")
validation_results = run_cross_dataset_validation(
    c_harm,
    f_harm,
    target_col='target',
    outer_splits=config['cv']['outer_folds'],
    inner_splits=config['cv']['inner_folds'],
    n_trials=config['cv']['optuna_n_trials'],
    random_state=config['random_state']
)

comp_df = validation_results['comparison_table']
print("\n=== Performance Transferability: Cleveland CV vs. Framingham External Validation ===")
display(comp_df)


Executing cross-dataset validation pipeline...
Running Cleveland-only nested CV on harmonized schema...


Fitting preprocessor and tuning hyper-parameters on full Cleveland dataset...


Evaluating trained pipeline on unseen Framingham dataset...



=== Performance Transferability: Cleveland CV vs. Framingham External Validation ===


,Model,Cleveland CV Accuracy,Framingham Ext Accuracy,Cleveland CV F1,Framingham Ext F1,Cleveland CV ROC-AUC,Framingham Ext ROC-AUC
0,RANDOM_FOREST,0.6702,0.7465,0.6299,0.3269,0.7190,0.6636
1,XGBOOST,0.6733,0.8481,0.6159,0.0000,0.7162,0.6583
2,ADABOOST,0.6533,0.7512,0.6189,0.3118,0.7055,0.6748
3,ENSEMBLE,0.6766,0.7717,0.6337,0.3056,0.7273,0.6728


## 3. Confusion Matrices on External Framingham Dataset
Plot confusion matrices across all models evaluated on the external Framingham cohort.


In [4]:
fram_metrics = validation_results['framingham_external']

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for idx, (m_name, m_data) in enumerate(fram_metrics.items()):
    cm = np.array(m_data['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[idx],
                xticklabels=['No CHD', '10-Yr CHD'],
                yticklabels=['No CHD', '10-Yr CHD'])
    axes[idx].set_title(f"Framingham Confusion Matrix: {m_name.upper()}")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
results_dir = '../' + config['results_dir']
plt.savefig(f"{results_dir}/framingham_external_confusion_matrices.png", dpi=300)
plt.show()
